<a href="https://colab.research.google.com/github/kashishpal26/Kashish-portfolio/blob/main/data_auditor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [3]:
!pip install opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/datasets/ealaxi/paysim1")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: kashishpal26
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1


100%|██████████| 178M/178M [00:01<00:00, 182MB/s]


In [4]:
import pandas as pd
file_path = 'paysim1/PS_20174392719_1491204439457_log.csv'
df= pd.read_csv(file_path, nrows = 50000)
print (df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [10]:
def run_data_audit(df):
  print("----Data Integrity Report----")

  null_counts = df.isnull().sum()
  print(f"\n[1] Missing Values Found:\n{null_counts[null_counts > 0]}")

  dup_count = df.duplicated().sum()
  print(f"\n[2] Duplicate Transactions: {dup_count}")

  negative_trans = df[df['amount'] < 0].shape[0]
  print(f"\n[3] Negative Transaction Entries: {negative_trans}")

  df['diff_check'] = (df['oldbalanceOrg'] - df['amount']).round(2) == df['newbalanceOrig'].round(2)
  inconsistent_balances = df[~df['diff_check']].shape[0]
  print(f"\n[4] Balance Inconsistency Errors: {inconsistent_balances}")
  print("----------------------------------")
run_data_audit(df)

----Data Integrity Report----

[1] Missing Values Found:
Series([], dtype: int64)

[2] Duplicate Transactions: 0

[3] Negative Transaction Entries: 0

[4] Balance Inconsistency Errors: 36118
----------------------------------


In [16]:
def generate_audit_log(df, df_cleaned):
  dropped_rows = df.index.difference(df_cleaned.index)
  print(f"Audit Trail: {len(dropped_rows)} entries flagged and moved to quarantine.")
  return dropped_rows

In [13]:
def clean_financial_data(df):
  df_cleaned= df.copy()
  df_cleaned= df_cleaned.fillna(0)
  df_cleaned = df_cleaned[df_cleaned['amount'] >= 0]
  df_cleaned = df_cleaned[
      (df_cleaned['oldbalanceOrg']-df_cleaned['amount']).round(2)== df_cleaned['newbalanceOrig'].round(2)
  ]

  print(f"Cleaning complete. Original rows: {len(df)}, Final rows: {len(df_cleaned)}")
  return df_cleaned
df_cleaned = clean_financial_data(df)

Cleaning complete. Original rows: 50000, Final rows: 13882


In [14]:
trace_report = {
    "Project_Name": "T.R.A.C.E.",
    "Full_Name": "Transactional Reconciliation & Audit Cleanup Engine",
    "Total_Rows_Processed": len(df),
    "Rows_Sanitized": len(df) - len(df_cleaned),
    "Data_Integrity_Score": f"{(len(df_cleaned)/len(df))*100:.2f}%",
    "Audit_Status": "Pass - Integrity Verified" if (len(df_cleaned)/len(df)) > 0.95 else "Flagged - Requires Review"
}

In [15]:
import json
print(json.dumps(trace_report, indent=4))

{
    "Project_Name": "T.R.A.C.E.",
    "Full_Name": "Transactional Reconciliation & Audit Cleanup Engine",
    "Total_Rows_Processed": 50000,
    "Rows_Sanitized": 36118,
    "Data_Integrity_Score": "27.76%",
    "Audit_Status": "Flagged - Requires Review"
}


In [17]:
import pandas as pd
import os

def run_trace_pipeline(df):
    quarantine = pd.DataFrame()
    df_cleaned = df.copy()

    mask_negative = df_cleaned['amount'] < 0

    mask_math_error = (df_cleaned['oldbalanceOrg'] - df_cleaned['amount']).round(2) != df_cleaned['newbalanceOrig'].round(2)

    corrupt_mask = mask_negative | mask_math_error

    quarantine = df_cleaned[corrupt_mask].copy()
    quarantine['error_reason'] = 'Invalid Amount or Math Mismatch'

    df_cleaned = df_cleaned[~corrupt_mask]

    # 4. Save results (Simulating professional output)
    quarantine.to_csv('quarantine_report.csv', index=False)

    print(f"Pipeline Execution Complete.")
    print(f"Rows Processed: {len(df)}")
    print(f"Rows Passed: {len(df_cleaned)}")
    print(f"Rows Quarantined: {len(quarantine)}")

    return df_cleaned, quarantine

df_final, df_quarantine = run_trace_pipeline(df)

Pipeline Execution Complete.
Rows Processed: 50000
Rows Passed: 13882
Rows Quarantined: 36118


In [18]:
import sqlite3

conn = sqlite3.connect(':memory:')

df.to_sql('transactions', conn, index=False)

query = "SELECT * FROM transactions WHERE amount > 10000 AND isFraud = 1"
fraud_report = pd.read_sql_query(query, conn)

print(f"High-value fraud cases found: {len(fraud_report)}")

High-value fraud cases found: 90
